<a href="https://colab.research.google.com/github/mustafayoruk/forest_ecosystem_regression/blob/main/forest_ecosystem_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.stats.api as sms
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 1. DATA EXTRACTION & CLEANING
def extract_totals_from_excel(filepath, sheet_name, value_col_name):
    """
    Reads an Excel sheet without headers, filters for valid years,
    cleans whitespace from numbers, and returns a structured DataFrame.
    """
    df_raw = pd.read_excel(filepath, sheet_name=sheet_name, header=None)
    data = []

    for index, row in df_raw.iterrows():
        val0 = str(row[0]).strip()
        # Check if the first column represents a valid year (e.g., 1973.0, 2020.0)
        if val0.startswith(('19', '20')) and len(val0) >= 4:
            try:
                year = int(float(val0))
                # Clean spaces from large numbers (e.g., "22 933 000" -> 22933000)
                val1 = str(row[1]).replace(' ', '').strip()
                total_val = float(val1)
                data.append({'Year': year, value_col_name: total_val})
            except (ValueError, TypeError):
                continue

    return pd.DataFrame(data)

# File path (Ensure this matches your repository's file structure)
excel_file = "forest_data_2024.xls"

# Extracting data from specific sheets
df_area = extract_totals_from_excel(excel_file, 'Forest area', 'ForestArea_Hectare')
df_stock = extract_totals_from_excel(excel_file, 'Growing stock', 'GrowingStock_m3')

# Merging datasets on the 'Year' column
df = df_area.merge(df_stock, on='Year', how='inner')
df = df.sort_values('Year').reset_index(drop=True)


# 2. MODEL BUILDING (ORDINARY LEAST SQUARES)
X = df['ForestArea_Hectare']
y = df['GrowingStock_m3']

# Adding a constant (intercept) for the Statsmodels OLS
X_with_const = sm.add_constant(X)

# Fitting the model
model = sm.OLS(y, X_with_const).fit()
predictions = model.predict(X_with_const)
residuals = model.resid

# 3. EVALUATION METRICS & ASSUMPTION TESTING
rmse = np.sqrt(mean_squared_error(y, predictions))
mae = mean_absolute_error(y, predictions)

print("--- MODEL PERFORMANCE ---")
print(f"R-Squared: {model.rsquared:.4f}")
print(f"Adjusted R-Squared: {model.rsquared_adj:.4f}")
print(f"RMSE: {rmse:,.2f}")
print(f"MAE: {mae:,.2f}\n")

print("--- ASSUMPTION TESTS ---")
# 1. Normality Test (Shapiro-Wilk)
shapiro_stat, shapiro_p = stats.shapiro(residuals)
print(f"Shapiro-Wilk p-value (Normality): {shapiro_p:.4f}")

# 2. Heteroscedasticity Test (Breusch-Pagan)
bp_stat, bp_p, _, _ = sms.het_breuschpagan(residuals, model.model.exog)
print(f"Breusch-Pagan p-value (Heteroscedasticity): {bp_p:.4f}")

# 3. Autocorrelation Test (Durbin-Watson)
dw_stat = sm.stats.stattools.durbin_watson(residuals)
print(f"Durbin-Watson Stat (Autocorrelation): {dw_stat:.4f}")

--- MODEL PERFORMANCE ---
R-Squared: 0.9644
Adjusted R-Squared: 0.9614
RMSE: 46,265,157.33
MAE: 36,581,305.40

--- ASSUMPTION TESTS ---
Shapiro-Wilk p-value (Normality): 0.4463
Breusch-Pagan p-value (Heteroscedasticity): 0.0389
Durbin-Watson Stat (Autocorrelation): 0.7318
